In [ ]:
# STEP 0: Mount Google Drive
import os
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
    project_path = '/content/gdrive/MyDrive/feral-cat-fcm'
except ImportError:
    project_path = os.path.abspath('feral-cat-fcm')  # running locally

# Define dataset and output paths
dataset_path = f'{project_path}/data/yolo_25-15-60'
output_path = f'{project_path}/outputs/yolo_25-15-60'
crop_output_path = f'{project_path}/outputs/crops/auto'

In [ ]:
# STEP 1: Install YOLOv12 via Ultralytics
!pip install -U ultralytics

In [ ]:
# Functions to save crops and predictions with ground truth overlay
from pathlib import Path
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches

class_names = ['back-leg', 'body', 'front-leg', 'tail']
class_colors = {0: 'blue', 1: 'red', 2: 'yellow', 3: 'green'}


def save_crops(result, image, crop_path, cat=''):
    name = Path(result.path).stem
    boxes = result.boxes
    for i, (box, conf, cls_id) in enumerate(zip(boxes.xyxy.tolist(), boxes.conf.tolist(), boxes.cls.int().tolist())):
        folder = Path(crop_path, class_names[cls_id], cat)
        folder.mkdir(parents=True, exist_ok=True)
        image.crop(tuple(map(int, box))).save(folder / f'{name}_crop_{i}_cls{cls_id}_conf{conf:.2f}.jpg')


def save_comparison(result, image, labels_path, output_path):
    w, h = image.size
    name = Path(result.path).stem
    fig, ax = plt.subplots(1)
    ax.imshow(image)

    # Predictions (solid)
    for (x1, y1, x2, y2), cls_id in zip(result.boxes.xyxy.tolist(), result.boxes.cls.int().tolist()):
        color = class_colors[cls_id]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=color, facecolor='none'))
        ax.text(x1, y1 - 5, f'Pred: {class_names[cls_id]}', color=color, fontsize=9, weight='bold')

    # Ground truth (dashed)
    label_file = Path(labels_path, f'{name}.txt')
    if label_file.exists():
        for line in label_file.read_text().splitlines():
            if not line.strip():
                continue
            cls_id, x_c, y_c, bw, bh = map(float, line.split())
            color = class_colors[int(cls_id)]
            x1, y1 = (x_c - bw / 2) * w, (y_c - bh / 2) * h
            ax.add_patch(patches.Rectangle((x1, y1), bw * w, bh * h, linewidth=2.5, edgecolor=color,
                                           facecolor='none', linestyle='dashed'))
            ax.text(x1, y1 + bh * h + 10, f'GT: {class_names[int(cls_id)]}', color=color, fontsize=9, weight='bold')

    ax.axis('off')
    Path(output_path).mkdir(parents=True, exist_ok=True)
    fig.savefig(Path(output_path, f'{name}_compare.jpg'), bbox_inches='tight', pad_inches=0)
    plt.close(fig)

In [ ]:
# STEP 2: Train YOLOv12n
from ultralytics import YOLO

model = YOLO('yolo12n.pt')
model.train(
    data=f'{dataset_path}/data.yaml',
    epochs=20,
    imgsz=640,
    batch=16,
    project=output_path,
    name='train',
    exist_ok=True
)

In [ ]:
# STEP 3: Run inference on each cat's test set and save crops
cats = sorted(d.name for d in Path(dataset_path, 'test').iterdir() if d.is_dir())

for cat in cats:
    results = model.predict(
        source=f'{dataset_path}/test/{cat}/images',
        conf=0.4,
        imgsz=640,
        save=True,
        project=f'{output_path}/predict',
        name=cat,
        exist_ok=True,
        stream=True
    )
    for result in results:
        image = Image.open(result.path).convert('RGB')
        save_crops(result, image, crop_output_path, cat)
        save_comparison(result, image, f'{dataset_path}/test/{cat}/labels', f'{output_path}/predict_with_gt/{cat}')
    print(f'Finished {cat}')